# Module 06 — Contrats JSON Schema + ADR

Valider bronze/silver avant et après écriture dans le lake.

In [1]:
import json
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

CONTRACTS = ROOT / "contracts"
print("Schémas:", list(CONTRACTS.glob("*.schema.json")))

Schémas: [PosixPath('/home/anthony-marais/Documents/data_project/contracts/silver.v1.schema.json'), PosixPath('/home/anthony-marais/Documents/data_project/contracts/bronze.v1.schema.json')]


## Étape 1 — Charger un exemple et valider

In [2]:
from presslake.contracts.validate import validate_bronze, validate_silver

bronze = json.loads((CONTRACTS / "examples" / "bronze.sample.json").read_text())
silver = json.loads((CONTRACTS / "examples" / "silver.sample.json").read_text())

validate_bronze(bronze)
validate_silver(silver)
print("Exemples OK")

Exemples OK


## Étape 2 — Voir une erreur de contrat (volontaire)

In [3]:
from presslake.contracts.validate import ContractValidationError

bad = dict(bronze)
del bad["content_hash"]

try:
    validate_bronze(bad)
except ContractValidationError as e:
    print(e)

Contrat bronze invalide (bronze.v1.schema.json):
  - (racine): 'content_hash' is a required property


## Étape 3 — Valider un objet réel depuis MinIO

In [4]:
from presslake.catalog.articles import list_articles_by_status
from presslake.storage.postgres import get_connection
from presslake.storage.s3 import get_json_object, get_s3_client, parse_s3_uri

with get_connection() as conn:
    arts = list_articles_by_status(conn, "parsed", limit=1)

if arts:
    uri = arts[0]["silver_s3_uri"]
    b, k = parse_s3_uri(uri)
    doc = get_json_object(get_s3_client(), b, k)
    validate_silver(doc)
    print("Silver lake OK:", doc["title"][:50])

Silver lake OK: 🔴 L'ancien Premier ministre Édouard Balladur est m


## Étape 4 — CLI

```bash
uv run presslake validate examples
uv run presslake validate lake --limit 5
```

ADR : `adr/0001-*.md`, `adr/0002-*.md`